# Lantern candidate RGB vetting — hybrid multi-loci v7

This is the checked/corrected vetting notebook (previous name `lantern_candidate_rgb_vetting_hybrid_v7(1)`) for the current hybrid multi-locus workflow.

It:

1. reads the **v8 candidate-analysis filtered catalog**;
2. reads the **v3 Second-Pass target membership and candidate alerts**;
3. cross-matches candidates to `lenses.csv`;
4. computes alert-level and target-level morphology diagnostics:
   - DiaSource `extendedness`;
   - `|apFlux - psfFlux| / |psfFlux|`;
   - DiaSource and PSF ellipticities from `ixx`, `iyy`, `ixy`;
5. applies the current vetting cuts:
   - median extendedness;
   - median absolute fractional aperture-vs-PSF flux difference;
6. shows diagnostic distributions **before** those cuts;
7. shows **all known-lens matches**, whether or not they survive the new cuts;
8. retrieves **ZIR-only** DP2 deep-coadd cutouts with a persistent in-memory cache;
9. defaults to **no bulk PDF generation** to avoid SIA/SODA rate-limit errors.

The candidate cutout width is 7.2 arcsec (`CUTOUT_RADIUS = 0.001 deg`).

In [ ]:
import ast
import io
import json
from collections.abc import Mapping
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.visualization import AsinhStretch, ImageNormalize

from IPython.display import display, Markdown
from lsst.images.serialization import read_archive
from lsst.rsp import RSPDiscovery
from lsst.rsp.utils import get_pyvo_auth
from matplotlib.backends.backend_pdf import PdfPages
from pyvo.dal.adhoc import SodaQuery

%matplotlib inline

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 9,
})

## 1. Inputs

In [ ]:
# ============================================================
# INPUT FILES
# ============================================================

CANDIDATES_FILE = Path(
    "candidate_analysis_hybrid_multi_loci/"
    "second_pass_candidates_filtered.parquet"
)

SECOND_PASS_DIR = Path(
    "second_pass_output_hybrid_multi_loci"
)

TARGET_MEMBERS_FILE = (
    SECOND_PASS_DIR
    / "second_pass_target_members.parquet"
)

CANDIDATE_ALERTS_FILE = (
    SECOND_PASS_DIR
    / "second_pass_candidate_alerts.parquet"
)

DATA_DIR = Path(
    "antares_data_hybrid"
)

LOCI_DIR = DATA_DIR / "loci"

KNOWN_LENS_FILE = Path(
    "candidate_lenses.csv"
)

OUTPUT_DIR = Path(
    "lantern_candidate_rgb_vetting_hybrid_v7"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# KNOWN-LENS CROSS-MATCH
# ============================================================

KNOWN_RA_COL = "ra"
KNOWN_DEC_COL = "dec"

# Set these if your lenses.csv contains them.
KNOWN_NAME_COL = None
KNOWN_NOTE_COL = None
KNOWN_Z_COL = None

MATCH_RADIUS_ARCSEC = 5.0

# ============================================================
# VETTING CUTS
# ============================================================

SECOND_PASS_SCORE_MIN = 0.80
SECOND_PASS_SCORE_MAX = None

GAIA_BP_RP_MIN = None
GAIA_BP_RP_MAX = None

I_MAG_MIN = None
I_MAG_MAX = None

# Current morphology cuts.
# Missing measurements are retained rather than automatically rejected.
EXTENDEDNESS_MEDIAN_MIN = 0.40

# Per alert:
#     abs(apFlux - psfFlux) / abs(psfFlux)
# Candidate quantity:
#     median of the valid per-alert values
AP_PSF_ABS_FRACDIFF_MEDIAN_MIN = 0.02

REQUIRE_KEEP_FLAG = True

SORT_BY = "second_pass_score"
SORT_ASCENDING = False
MAX_CANDIDATES = None

# ============================================================
# DIAGNOSTIC DEFINITIONS
# ============================================================

AB_ZEROPOINT_NJY = 31.4

# Used only for reporting the fraction of high-extendedness alerts.
EXTENDEDNESS_REPORT_THRESHOLD = 0.5

# ============================================================
# DP2 ZIR CUTOUTS
# ============================================================

# 0.001 deg radius = 3.6 arcsec radius = 7.2 arcsec full width.
CUTOUT_RADIUS = 0.001

FIGSIZE = (4.8, 5.8)
ASINH_A = 0.002
VMIN = -0.03
VMAX = 3000

SKIP_MISSING = True

# General preview is off by default to avoid unnecessary DP2 requests.
START = 0
N_SHOW = 20
SHOW_PREVIEW = True #False

# Known-lens cutouts are shown regardless of whether the new morphology
# cuts remove them, so they can be used to assess the cuts.
SHOW_MATCHED_CUTOUTS = True
MAX_MATCHED_TO_SHOW = None

SHOW_MATCH_DETAILS = True
MAX_MATCH_DETAILS = None
MAX_ALERT_ROWS = 30

# Bulk PDF generation is disabled by default because many synchronous
# SIA/SODA requests can trigger HTTP 429 rate limits.
SAVE_PDF = False
SAVE_PNG_PAGES = False
OVERWRITE_OUTPUT = True

## 2. Load the hybrid candidate catalog

In [ ]:
cand = pd.read_parquet(
    CANDIDATES_FILE
).copy()

required = {
    "target_id",
    "ra",
    "dec",
    "second_pass_score",
}

missing = required - set(cand.columns)
if missing:
    raise KeyError(
        "Candidate file does not match the hybrid target-level workflow. "
        f"Missing: {sorted(missing)}"
    )

if cand["target_id"].duplicated().any():
    raise ValueError(
        "Duplicate target_id rows found in the candidate catalog."
    )

cand["ra"] = pd.to_numeric(
    cand["ra"],
    errors="coerce",
)
cand["dec"] = pd.to_numeric(
    cand["dec"],
    errors="coerce",
)
cand["second_pass_score"] = pd.to_numeric(
    cand["second_pass_score"],
    errors="coerce",
)

if cand[["ra", "dec"]].isna().any(axis=None):
    raise ValueError(
        "Candidate catalog contains invalid RA/Dec values."
    )

cand = (
    cand
    .sort_values(
        SORT_BY,
        ascending=SORT_ASCENDING,
    )
    .reset_index(drop=True)
)

cand["rank"] = np.arange(
    1,
    len(cand) + 1,
)

ID_COL = "target_id"

print(f"Candidates loaded: {len(cand):,}")
print(f"ID column: {ID_COL}")

## 3. Load target membership, ANTARES loci, and candidate alerts

In [ ]:
target_members = pd.read_parquet(
    TARGET_MEMBERS_FILE
)

candidate_alerts = pd.read_parquet(
    CANDIDATE_ALERTS_FILE
)

required_members = {
    "target_id",
    "locus_id",
}

required_alerts = {
    "target_id",
    "locus_id",
    "alert_id",
    "mjd",
    "alert_properties",
}

missing = required_members - set(target_members.columns)
if missing:
    raise KeyError(
        "Target-membership file is missing: "
        f"{sorted(missing)}"
    )

missing = required_alerts - set(candidate_alerts.columns)
if missing:
    raise KeyError(
        "Candidate-alert file is missing: "
        f"{sorted(missing)}"
    )

if candidate_alerts["alert_id"].duplicated().any():
    raise ValueError(
        "Duplicate alert_id rows found in second_pass_candidate_alerts.parquet."
    )

candidate_target_ids = set(
    cand["target_id"]
)

relevant_members = target_members[
    target_members["target_id"].isin(
        candidate_target_ids
    )
].copy()

relevant_alerts = candidate_alerts[
    candidate_alerts["target_id"].isin(
        candidate_target_ids
    )
].copy()

missing_member_targets = (
    candidate_target_ids
    - set(relevant_members["target_id"])
)

missing_alert_targets = (
    candidate_target_ids
    - set(relevant_alerts["target_id"])
)

if missing_member_targets:
    raise RuntimeError(
        f"{len(missing_member_targets)} candidates have no target-member rows."
    )

if missing_alert_targets:
    raise RuntimeError(
        f"{len(missing_alert_targets)} candidates have no candidate-alert rows."
    )

member_locus_ids = (
    relevant_members["locus_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

locus_columns = [
    "locus_id",
    "ra",
    "dec",
    "tags",
    "catalogs",
    "num_tagged_alerts",
    "max_score",
    "ztf_object_id",
    "survey_lsst",
    "gaia_variability_class",
    "gaia_parallax_over_error",
    "gaia_pmdec",
    "gaia_pmdec_error",
    "gaia_pmra",
    "gaia_pmra_error",
    "gaia_in_qso_candidates",
    "gaia_phot_g_mean_mag",
    "gaia_phot_g_mean_flux_over_error",
    "gaia_bp_g",
    "gaia_g_rp",
    "milliquas_match",
]

loci_dataset = ds.dataset(
    LOCI_DIR,
    format="parquet",
)

available_locus_columns = set(
    loci_dataset.schema.names
)

load_locus_columns = [
    c
    for c in locus_columns
    if c in available_locus_columns
]

try:
    loci = (
        loci_dataset
        .to_table(
            columns=load_locus_columns,
            filter=ds.field("locus_id").isin(
                member_locus_ids
            ),
        )
        .to_pandas()
    )
except Exception as e:
    print(
        "Filtered locus load failed; falling back to an unfiltered "
        f"column load. Reason: {e}"
    )

    loci = (
        loci_dataset
        .to_table(
            columns=load_locus_columns
        )
        .to_pandas()
    )

    loci = loci[
        loci["locus_id"]
        .astype(str)
        .isin(member_locus_ids)
    ].copy()

if loci["locus_id"].duplicated().any():
    print(
        "Warning: duplicate locus_id rows found; keeping the first row."
    )
    loci = loci.drop_duplicates(
        "locus_id",
        keep="first",
    )

print(f"Target-member rows: {len(relevant_members):,}")
print(f"Candidate alerts:   {len(relevant_alerts):,}")
print(f"Member loci loaded: {len(loci):,}")

## 4. Cross-match to `lenses.csv`

In [ ]:
known = pd.read_csv(
    KNOWN_LENS_FILE
).copy()

for col in [
    KNOWN_RA_COL,
    KNOWN_DEC_COL,
]:
    if col not in known.columns:
        raise KeyError(
            f"Known-lens file is missing required column: {col}"
        )

known[KNOWN_RA_COL] = pd.to_numeric(
    known[KNOWN_RA_COL],
    errors="coerce",
)
known[KNOWN_DEC_COL] = pd.to_numeric(
    known[KNOWN_DEC_COL],
    errors="coerce",
)

known = (
    known
    .dropna(
        subset=[
            KNOWN_RA_COL,
            KNOWN_DEC_COL,
        ]
    )
    .reset_index(drop=True)
)

if len(known) == 0:
    raise ValueError(
        "No valid known-lens RA/Dec rows remain."
    )

cand_coord = SkyCoord(
    ra=cand["ra"].to_numpy() * u.deg,
    dec=cand["dec"].to_numpy() * u.deg,
)

known_coord = SkyCoord(
    ra=known[KNOWN_RA_COL].to_numpy() * u.deg,
    dec=known[KNOWN_DEC_COL].to_numpy() * u.deg,
)

idx, sep2d, _ = (
    cand_coord
    .match_to_catalog_sky(
        known_coord
    )
)

cand["known_lens_index"] = idx
cand["known_lens_sep_arcsec"] = sep2d.arcsec
cand["known_lens_match"] = (
    cand["known_lens_sep_arcsec"]
    <= MATCH_RADIUS_ARCSEC
)

if (
    KNOWN_NAME_COL is not None
    and KNOWN_NAME_COL in known.columns
):
    cand["known_lens_name"] = np.where(
        cand["known_lens_match"],
        known.iloc[idx][KNOWN_NAME_COL]
        .astype(str)
        .to_numpy(),
        "",
    )
else:
    cand["known_lens_name"] = np.where(
        cand["known_lens_match"],
        known.index.to_numpy()[idx].astype(str),
        "",
    )

if (
    KNOWN_NOTE_COL is not None
    and KNOWN_NOTE_COL in known.columns
):
    cand["known_lens_note"] = np.where(
        cand["known_lens_match"],
        known.iloc[idx][KNOWN_NOTE_COL]
        .astype(str)
        .to_numpy(),
        "",
    )

if (
    KNOWN_Z_COL is not None
    and KNOWN_Z_COL in known.columns
):
    known_z = pd.to_numeric(
        known.iloc[idx][KNOWN_Z_COL],
        errors="coerce",
    ).to_numpy()

    cand["known_lens_z"] = np.where(
        cand["known_lens_match"],
        known_z,
        np.nan,
    )

cand["legacy_viewer_url"] = [
    (
        "https://www.legacysurvey.org/viewer"
        f"?ra={ra:.8f}"
        f"&dec={dec:.8f}"
        "&layer=ls-dr11"
        "&zoom=16"
    )
    for ra, dec in zip(
        cand["ra"],
        cand["dec"],
    )
]

print(f"Known lenses loaded: {len(known):,}")
print(
    f"Candidate matches within {MATCH_RADIUS_ARCSEC:.1f}\" : "
    f"{int(cand['known_lens_match'].sum()):,}"
)

match_cols = [
    "rank",
    "target_id",
    "ra",
    "dec",
    "second_pass_score",
    "known_lens_match",
    "known_lens_name",
    "known_lens_sep_arcsec",
]

display(
    cand.loc[
        cand["known_lens_match"],
        match_cols,
    ]
)

## 5. Build DiaSource morphology / photometry diagnostics

In [ ]:
def as_dict(value):
    if isinstance(value, Mapping):
        return dict(value)

    if isinstance(value, str):
        for parser in (
            json.loads,
            ast.literal_eval,
        ):
            try:
                parsed = parser(value)
                if isinstance(parsed, Mapping):
                    return dict(parsed)
            except Exception:
                pass

    return {}


def get_prop(
    props,
    key,
    default=np.nan,
):
    props = as_dict(props)

    value = props.get(
        key,
        default,
    )

    if value is None:
        return default

    return value


def to_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def to_bool_or_nan(value):
    if value is None:
        return np.nan

    if isinstance(
        value,
        (
            bool,
            np.bool_,
        ),
    ):
        return bool(value)

    if isinstance(
        value,
        (
            int,
            np.integer,
            float,
            np.floating,
        ),
    ):
        if pd.isna(value):
            return np.nan
        return bool(value)

    text = str(value).strip().lower()

    if text in {
        "true",
        "t",
        "1",
        "yes",
    }:
        return True

    if text in {
        "false",
        "f",
        "0",
        "no",
    }:
        return False

    return np.nan


def compute_e_components(
    ixx,
    iyy,
    ixy,
):
    ixx = to_float(ixx)
    iyy = to_float(iyy)
    ixy = to_float(ixy)

    trace = ixx + iyy

    if not (
        np.isfinite(ixx)
        and np.isfinite(iyy)
        and np.isfinite(ixy)
        and np.isfinite(trace)
        and trace != 0
    ):
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    e1 = (
        (ixx - iyy)
        / trace
    )

    e2 = (
        2.0 * ixy
        / trace
    )

    e = np.sqrt(
        e1**2
        + e2**2
    )

    return (
        e1,
        e2,
        e,
    )


alerts = relevant_alerts.copy()

property_keys = {
    "band": "lsst_diaSource_band",
    "science_flux": "lsst_diaSource_scienceFlux",
    "science_flux_err": "lsst_diaSource_scienceFluxErr",
    "psf_flux": "lsst_diaSource_psfFlux",
    "psf_flux_err": "lsst_diaSource_psfFluxErr",
    "ap_flux": "lsst_diaSource_apFlux",
    "extendedness": "lsst_diaSource_extendedness",
    "shape_flag": "lsst_diaSource_shape_flag",
    "src_ixx": "lsst_diaSource_ixx",
    "src_iyy": "lsst_diaSource_iyy",
    "src_ixy": "lsst_diaSource_ixy",
    "psf_ixx": "lsst_diaSource_ixxPSF",
    "psf_iyy": "lsst_diaSource_iyyPSF",
    "psf_ixy": "lsst_diaSource_ixyPSF",
}

for out_col, key in property_keys.items():
    alerts[out_col] = alerts[
        "alert_properties"
    ].map(
        lambda p, k=key: get_prop(
            p,
            k,
        )
    )

numeric_cols = [
    c
    for c in property_keys
    if c not in {
        "band",
        "shape_flag",
    }
]

alerts[numeric_cols] = (
    alerts[numeric_cols]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

alerts["shape_flag"] = (
    alerts["shape_flag"]
    .map(
        to_bool_or_nan
    )
)

# -------------------------
# Science magnitude
# -------------------------

good_science_flux = (
    np.isfinite(
        alerts["science_flux"]
    )
    & (
        alerts["science_flux"] > 0
    )
)

alerts["science_mag"] = np.nan

alerts.loc[
    good_science_flux,
    "science_mag",
] = (
    AB_ZEROPOINT_NJY
    - 2.5
    * np.log10(
        alerts.loc[
            good_science_flux,
            "science_flux",
        ]
    )
)

# -------------------------
# Difference-image PSF S/N
# -------------------------

good_psf_err = (
    np.isfinite(
        alerts["psf_flux_err"]
    )
    & (
        alerts["psf_flux_err"] != 0
    )
)

alerts["diff_psf_snr"] = np.nan

alerts.loc[
    good_psf_err,
    "diff_psf_snr",
] = (
    alerts.loc[
        good_psf_err,
        "psf_flux",
    ]
    / alerts.loc[
        good_psf_err,
        "psf_flux_err",
    ]
)

alerts["abs_diff_psf_snr"] = np.abs(
    alerts["diff_psf_snr"]
)

# -------------------------
# Aperture-vs-PSF flux metric
# -------------------------

good_ap_psf = (
    np.isfinite(
        alerts["ap_flux"]
    )
    & np.isfinite(
        alerts["psf_flux"]
    )
    & (
        np.abs(
            alerts["psf_flux"]
        ) > 0
    )
)

alerts["ap_psf_abs_fracdiff"] = np.nan

alerts.loc[
    good_ap_psf,
    "ap_psf_abs_fracdiff",
] = (
    np.abs(
        alerts.loc[
            good_ap_psf,
            "ap_flux",
        ]
        - alerts.loc[
            good_ap_psf,
            "psf_flux",
        ]
    )
    / np.abs(
        alerts.loc[
            good_ap_psf,
            "psf_flux",
        ]
    )
)

# Keep the signed raw difference as a diagnostic too.
alerts["ap_psf_flux_diff"] = np.nan

good_raw_diff = (
    np.isfinite(
        alerts["ap_flux"]
    )
    & np.isfinite(
        alerts["psf_flux"]
    )
)

alerts.loc[
    good_raw_diff,
    "ap_psf_flux_diff",
] = (
    alerts.loc[
        good_raw_diff,
        "ap_flux",
    ]
    - alerts.loc[
        good_raw_diff,
        "psf_flux",
    ]
)

# -------------------------
# Source and PSF ellipticity
# -------------------------

src_e = alerts.apply(
    lambda row: compute_e_components(
        row["src_ixx"],
        row["src_iyy"],
        row["src_ixy"],
    ),
    axis=1,
    result_type="expand",
)

src_e.columns = [
    "src_e1",
    "src_e2",
    "src_e",
]

psf_e = alerts.apply(
    lambda row: compute_e_components(
        row["psf_ixx"],
        row["psf_iyy"],
        row["psf_ixy"],
    ),
    axis=1,
    result_type="expand",
)

psf_e.columns = [
    "psf_e1",
    "psf_e2",
    "psf_e",
]

alerts = pd.concat(
    [
        alerts,
        src_e,
        psf_e,
    ],
    axis=1,
)

alerts["delta_e"] = (
    alerts["src_e"]
    - alerts["psf_e"]
)

alerts["abs_delta_e"] = np.abs(
    alerts["delta_e"]
)

# -------------------------
# Validity mask for moment-based morphology
# -------------------------

# shape_flag=False means the shape measurement did not report a failure.
shape_good = (
    alerts["shape_flag"].isna()
    | (
        alerts["shape_flag"]
        == False
    )
)

alerts["extendedness_valid"] = (
    np.isfinite(
        alerts["extendedness"]
    )
    & shape_good
)

alerts["extendedness_for_summary"] = (
    alerts["extendedness"]
    .where(
        alerts["extendedness_valid"]
    )
)

alerts["delta_e_for_summary"] = (
    alerts["delta_e"]
    .where(
        shape_good
    )
)

alerts["abs_delta_e_for_summary"] = (
    alerts["abs_delta_e"]
    .where(
        shape_good
    )
)

print("Alert-level diagnostic availability:")
for col in [
    "extendedness_for_summary",
    "ap_psf_abs_fracdiff",
    "delta_e_for_summary",
]:
    print(
        f"  {col}: "
        f"{alerts[col].notna().sum():,} / {len(alerts):,}"
    )

## 6. Build one diagnostic summary per candidate target

In [ ]:
def finite_median(series):
    values = pd.to_numeric(
        series,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan

    return float(
        np.median(values)
    )


def finite_max(series):
    values = pd.to_numeric(
        series,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan

    return float(
        np.max(values)
    )


def frac_gt_extendedness_threshold(series):
    values = pd.to_numeric(
        series,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan

    return float(
        (
            values
            > EXTENDEDNESS_REPORT_THRESHOLD
        ).mean()
    )


candidate_diag = (
    alerts
    .groupby(
        "target_id"
    )
    .agg(
        diagnostic_n_alerts=(
            "alert_id",
            "size",
        ),

        extendedness_n=(
            "extendedness_for_summary",
            "count",
        ),
        extendedness_median=(
            "extendedness_for_summary",
            finite_median,
        ),
        extendedness_max=(
            "extendedness_for_summary",
            finite_max,
        ),
        extendedness_frac_gt_0p5=(
            "extendedness_for_summary",
            frac_gt_extendedness_threshold,
        ),

        ap_psf_abs_fracdiff_n=(
            "ap_psf_abs_fracdiff",
            "count",
        ),
        ap_psf_abs_fracdiff_median=(
            "ap_psf_abs_fracdiff",
            finite_median,
        ),

        ap_psf_flux_diff_median=(
            "ap_psf_flux_diff",
            finite_median,
        ),

        delta_e_n=(
            "delta_e_for_summary",
            "count",
        ),
        delta_e_median=(
            "delta_e_for_summary",
            finite_median,
        ),
        abs_delta_e_median=(
            "abs_delta_e_for_summary",
            finite_median,
        ),
        abs_delta_e_max=(
            "abs_delta_e_for_summary",
            finite_max,
        ),

        median_science_mag=(
            "science_mag",
            finite_median,
        ),
        median_abs_diff_psf_snr=(
            "abs_diff_psf_snr",
            finite_median,
        ),
        max_abs_diff_psf_snr=(
            "abs_diff_psf_snr",
            finite_max,
        ),
    )
    .reset_index()
)

cand = cand.merge(
    candidate_diag,
    on="target_id",
    how="left",
    validate="one_to_one",
)

print(
    "Candidate-level diagnostics constructed: "
    f"{len(candidate_diag):,}"
)

display(
    cand[
        [
            "rank",
            "target_id",
            "second_pass_score",
            "extendedness_median",
            "extendedness_n",
            "ap_psf_abs_fracdiff_median",
            "ap_psf_abs_fracdiff_n",
            "delta_e_median",
            "delta_e_n",
            "known_lens_match",
        ]
    ].head(20)
)

## 7. Diagnostic distributions before the new morphology cuts

In [ ]:
# ------------------------------------------------------------
# Alert-level extendedness
# ------------------------------------------------------------

x = (
    alerts[
        "extendedness_for_summary"
    ]
    .dropna()
    .to_numpy(
        dtype=float
    )
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

if len(x):
    ax.hist(
        x,
        bins=np.linspace(
            0,
            1,
            41,
        ),
        histtype="step",
    )

ax.axvline(
    EXTENDEDNESS_MEDIAN_MIN,
    color="black",
    ls="--",
)

ax.set_xlabel(
    "DiaSource extendedness"
)
ax.set_ylabel(
    "Alerts"
)

fig.tight_layout()
plt.show()


# ------------------------------------------------------------
# Candidate median extendedness
# ------------------------------------------------------------

x = (
    cand[
        "extendedness_median"
    ]
    .dropna()
    .to_numpy(
        dtype=float
    )
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

if len(x):
    ax.hist(
        x,
        bins=np.linspace(
            0,
            1,
            41,
        ),
        histtype="step",
    )

ax.axvline(
    EXTENDEDNESS_MEDIAN_MIN,
    color="black",
    ls="--",
)

ax.set_xlabel(
    "Candidate median extendedness"
)
ax.set_ylabel(
    "Candidates"
)

fig.tight_layout()
plt.show()


# ------------------------------------------------------------
# Candidate median absolute fractional ap-vs-PSF difference
# ------------------------------------------------------------

x = (
    cand[
        "ap_psf_abs_fracdiff_median"
    ]
    .dropna()
    .to_numpy(
        dtype=float
    )
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

if len(x):
    # Clip only for the displayed range; the values themselves are unchanged.
    xmax = max(
        AP_PSF_ABS_FRACDIFF_MEDIAN_MIN * 3,
        np.nanpercentile(
            x,
            95,
        ),
    )

    xmax = min(
        xmax,
        np.nanmax(x),
    )

    ax.hist(
        x[
            x <= xmax
        ],
        bins=40,
        histtype="step",
    )

ax.axvline(
    AP_PSF_ABS_FRACDIFF_MEDIAN_MIN,
    color="black",
    ls="--",
)

ax.set_xlabel(
    r"Median $|F_{\rm ap}-F_{\rm PSF}|/|F_{\rm PSF}|$"
)
ax.set_ylabel(
    "Candidates"
)

fig.tight_layout()
plt.show()


# ------------------------------------------------------------
# Extendedness vs fractional flux metric
# ------------------------------------------------------------

good = (
    cand["extendedness_median"].notna()
    & cand[
        "ap_psf_abs_fracdiff_median"
    ].notna()
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

if good.sum():
    if good.sum() >= 300:
        hb = ax.hexbin(
            cand.loc[
                good,
                "extendedness_median",
            ],
            cand.loc[
                good,
                "ap_psf_abs_fracdiff_median",
            ],
            gridsize=100,
            bins="log",
            mincnt=1,
        )

        cbar = fig.colorbar(
            hb,
            ax=ax,
        )
        cbar.set_label(
            "Candidates"
        )

    else:
        ax.scatter(
            cand.loc[
                good,
                "extendedness_median",
            ],
            cand.loc[
                good,
                "ap_psf_abs_fracdiff_median",
            ],
            s=4,
            alpha=0.25,
        )

known_good = (
    good
    & cand["known_lens_match"]
)

if known_good.sum():
    ax.scatter(
        cand.loc[
            known_good,
            "extendedness_median",
        ],
        cand.loc[
            known_good,
            "ap_psf_abs_fracdiff_median",
        ],
        s=30,
        facecolors="none",
        edgecolors="red",
        linewidths=1.0,
        label="Known-lens match",
    )

    ax.legend(
        frameon=False,
    )

ax.axvline(
    EXTENDEDNESS_MEDIAN_MIN,
    color="r",
    ls="--",
)

ax.axhline(
    AP_PSF_ABS_FRACDIFF_MEDIAN_MIN,
    color="r",
    ls="--",
)

ax.set_xlabel(
    "Median extendedness"
)
ax.set_ylabel(
    r"Median $|F_{\rm ap}-F_{\rm PSF}|/|F_{\rm PSF}|$"
)
ax.set_yscale('log')
fig.tight_layout()
plt.show()

## 8. Apply candidate-vetting cuts

In [ ]:
selected = cand.copy()

n_start = len(selected)

def report_cut(
    name,
    before,
    after,
):
    print(
        f"{name:<42s}: "
        f"{before:6,d} -> {after:6,d} "
        f"(removed {before-after:,})"
    )


if SECOND_PASS_SCORE_MIN is not None:
    before = len(selected)

    selected = selected[
        selected["second_pass_score"]
        >= SECOND_PASS_SCORE_MIN
    ]

    report_cut(
        "Second-Pass minimum",
        before,
        len(selected),
    )


if SECOND_PASS_SCORE_MAX is not None:
    before = len(selected)

    selected = selected[
        selected["second_pass_score"]
        <= SECOND_PASS_SCORE_MAX
    ]

    report_cut(
        "Second-Pass maximum",
        before,
        len(selected),
    )


if (
    GAIA_BP_RP_MIN is not None
    and "gaia_bp_rp_max" in selected.columns
):
    before = len(selected)

    selected = selected[
        selected["gaia_bp_rp_max"].isna()
        | (
            selected["gaia_bp_rp_max"]
            >= GAIA_BP_RP_MIN
        )
    ]

    report_cut(
        "Gaia BP-RP minimum",
        before,
        len(selected),
    )


if (
    GAIA_BP_RP_MAX is not None
    and "gaia_bp_rp_max" in selected.columns
):
    before = len(selected)

    selected = selected[
        selected["gaia_bp_rp_max"].isna()
        | (
            selected["gaia_bp_rp_max"]
            <= GAIA_BP_RP_MAX
        )
    ]

    report_cut(
        "Gaia BP-RP maximum",
        before,
        len(selected),
    )


if (
    I_MAG_MIN is not None
    and "i_mag" in selected.columns
):
    before = len(selected)

    selected = selected[
        selected["i_mag"].isna()
        | (
            selected["i_mag"]
            >= I_MAG_MIN
        )
    ]

    report_cut(
        "i magnitude minimum",
        before,
        len(selected),
    )


if (
    I_MAG_MAX is not None
    and "i_mag" in selected.columns
):
    before = len(selected)

    selected = selected[
        selected["i_mag"].isna()
        | (
            selected["i_mag"]
            <= I_MAG_MAX
        )
    ]

    report_cut(
        "i magnitude maximum",
        before,
        len(selected),
    )


if EXTENDEDNESS_MEDIAN_MIN is not None:
    before = len(selected)

    selected = selected[
        selected["extendedness_median"].isna()
        | (
            selected["extendedness_median"]
            >= EXTENDEDNESS_MEDIAN_MIN
        )
    ]

    report_cut(
        "Median extendedness maximum",
        before,
        len(selected),
    )


if AP_PSF_ABS_FRACDIFF_MEDIAN_MIN is not None:
    before = len(selected)

    selected = selected[
        selected[
            "ap_psf_abs_fracdiff_median"
        ].isna()
        | (
            selected[
                "ap_psf_abs_fracdiff_median"
            ]
            >= AP_PSF_ABS_FRACDIFF_MEDIAN_MIN
        )
    ]

    report_cut(
        "Median |ap-PSF|/|PSF| maximum",
        before,
        len(selected),
    )


if (
    REQUIRE_KEEP_FLAG
    and "keep_flag" in selected.columns
):
    before = len(selected)

    selected = selected[
        selected["keep_flag"]
    ]

    report_cut(
        "Previous keep_flag",
        before,
        len(selected),
    )


if MAX_CANDIDATES is not None:
    selected = selected.head(
        MAX_CANDIDATES
    )


selected = (
    selected
    .sort_values(
        SORT_BY,
        ascending=SORT_ASCENDING,
    )
    .reset_index(drop=True)
)

selected_ids = set(
    selected["target_id"]
)

cand["passes_vetting"] = (
    cand["target_id"]
    .isin(
        selected_ids
    )
)

print()
print(
    f"Final selected candidates: {len(selected):,} / {n_start:,}"
)

n_known_all = int(
    cand["known_lens_match"].sum()
)

n_known_selected = int(
    (
        cand["known_lens_match"]
        & cand["passes_vetting"]
    ).sum()
)

print(
    "Known-lens matches surviving vetting: "
    f"{n_known_selected:,} / {n_known_all:,}"
)

display(
    selected[
        [
            c
            for c in [
                "rank",
                "target_id",
                "second_pass_score",
                "gaia_bp_rp_max",
                "i_mag",
                "extendedness_median",
                "extendedness_n",
                "ap_psf_abs_fracdiff_median",
                "ap_psf_abs_fracdiff_n",
                "delta_e_median",
                "abs_delta_e_median",
                "known_lens_match",
                "known_lens_name",
                "known_lens_sep_arcsec",
            ]
            if c in selected.columns
        ]
    ].head(30)
)

## 9. Save the selected candidate table

In [ ]:
selected_csv = (
    OUTPUT_DIR
    / "selected_candidates.csv"
)

selected_parquet = (
    OUTPUT_DIR
    / "selected_candidates.parquet"
)

existing = [
    path
    for path in [
        selected_csv,
        selected_parquet,
    ]
    if path.exists()
]

if existing and not OVERWRITE_OUTPUT:
    raise FileExistsError(
        "Output files already exist and OVERWRITE_OUTPUT=False: "
        f"{existing}"
    )

selected.to_csv(
    selected_csv,
    index=False,
)

selected.to_parquet(
    selected_parquet,
    index=False,
)

print("Written:")
print(" ", selected_csv)
print(" ", selected_parquet)

## 10. Helpers for member-locus and alert-level inspection

In [ ]:
def get_target_member_ids(
    target_id,
):
    return (
        relevant_members.loc[
            relevant_members["target_id"]
            == target_id,
            "locus_id",
        ]
        .astype(str)
        .tolist()
    )


def get_target_locus_info(
    target_id,
):
    member_ids = get_target_member_ids(
        target_id
    )

    out = loci[
        loci["locus_id"]
        .astype(str)
        .isin(member_ids)
    ].copy()

    order = {
        locus_id: i
        for i, locus_id in enumerate(
            member_ids
        )
    }

    if len(out):
        out["_order"] = (
            out["locus_id"]
            .astype(str)
            .map(order)
        )

        out = (
            out
            .sort_values("_order")
            .drop(columns="_order")
        )

    return out


def locus_summary_text(
    row,
):
    locus_info = get_target_locus_info(
        row["target_id"]
    )

    if len(locus_info) == 0:
        if "member_locus_ids" in row.index:
            return (
                "loci="
                f"{row['member_locus_ids']}"
            )
        return "loci=unavailable"

    parts = []

    for _, loc in locus_info.iterrows():
        locus_id = str(
            loc["locus_id"]
        )

        max_score = loc.get(
            "max_score",
            np.nan,
        )

        n_tagged = loc.get(
            "num_tagged_alerts",
            np.nan,
        )

        score_text = (
            f"{float(max_score):.3f}"
            if pd.notna(max_score)
            else "nan"
        )

        tagged_text = (
            str(int(n_tagged))
            if pd.notna(n_tagged)
            else "nan"
        )

        parts.append(
            f"{locus_id} "
            f"(first={score_text}, tagged={tagged_text})"
        )

    return (
        "loci: "
        + "; ".join(parts)
    )


def get_target_alert_detail(
    target_id,
):
    return (
        alerts[
            alerts["target_id"]
            == target_id
        ]
        .sort_values("mjd")
        .copy()
    )

## 11. Detailed information for every known-lens match

In [ ]:
def show_matched_target_details(
    row,
    max_alert_rows=MAX_ALERT_ROWS,
):
    target_id = row["target_id"]

    passed = bool(
        row.get(
            "passes_vetting",
            False,
        )
    )

    display(
        Markdown(
            f"### Rank {int(row['rank'])}: `{target_id}` "
            f"— passes current vetting: **{passed}**"
        )
    )

    display(
        Markdown(
            f"[Open Legacy Survey viewer]({row['legacy_viewer_url']})"
        )
    )

    target_cols = [
        c
        for c in [
            "rank",
            "target_id",
            "ra",
            "dec",
            "second_pass_score",
            "first_cut_max_score",
            "n_member_loci",
            "member_locus_ids",
            "n_tagged",
            "n_detections",
            "n_lsst_alerts",
            "percent_tagged",
            "n_alerts",
            "n_bands",
            "u_mag",
            "g_mag",
            "r_mag",
            "i_mag",
            "z_mag",
            "y_mag",
            "gaia_bp_rp_max",
            "gaia_phot_g_mean_mag_brightest",
            "gaia_in_qso_candidates_any",
            "milliquas_match",
            "extendedness_n",
            "extendedness_median",
            "extendedness_max",
            "extendedness_frac_gt_0p5",
            "ap_psf_abs_fracdiff_n",
            "ap_psf_abs_fracdiff_median",
            "delta_e_n",
            "delta_e_median",
            "abs_delta_e_median",
            "abs_delta_e_max",
            "known_lens_name",
            "known_lens_sep_arcsec",
            "known_lens_z",
            "known_lens_note",
            "passes_vetting",
        ]
        if c in row.index
    ]

    summary = pd.DataFrame(
        {
            "value": [
                row[c]
                for c in target_cols
            ]
        },
        index=target_cols,
    )

    display(summary)

    display(
        Markdown(
            "**ANTARES member loci**"
        )
    )

    locus_info = get_target_locus_info(
        target_id
    )

    locus_cols = [
        c
        for c in [
            "locus_id",
            "ra",
            "dec",
            "max_score",
            "num_tagged_alerts",
            "tags",
            "catalogs",
            "ztf_object_id",
            "survey_lsst",
            "milliquas_match",
            "gaia_in_qso_candidates",
            "gaia_phot_g_mean_mag",
            "gaia_bp_g",
            "gaia_g_rp",
            "gaia_parallax_over_error",
            "gaia_pmra",
            "gaia_pmra_error",
            "gaia_pmdec",
            "gaia_pmdec_error",
        ]
        if c in locus_info.columns
    ]

    if len(locus_info):
        display(
            locus_info[
                locus_cols
            ]
        )

    display(
        Markdown(
            "**LSST DiaSource alert summary by band**"
        )
    )

    detail = get_target_alert_detail(
        target_id
    )

    band_summary = (
        detail
        .groupby(
            "band",
            dropna=False,
        )
        .agg(
            n_alerts=(
                "alert_id",
                "size",
            ),
            first_mjd=(
                "mjd",
                "min",
            ),
            last_mjd=(
                "mjd",
                "max",
            ),
            extendedness_median=(
                "extendedness_for_summary",
                finite_median,
            ),
            extendedness_max=(
                "extendedness_for_summary",
                finite_max,
            ),
            ap_psf_abs_fracdiff_median=(
                "ap_psf_abs_fracdiff",
                finite_median,
            ),
            delta_e_median=(
                "delta_e_for_summary",
                finite_median,
            ),
            abs_delta_e_median=(
                "abs_delta_e_for_summary",
                finite_median,
            ),
            median_abs_diff_psf_snr=(
                "abs_diff_psf_snr",
                finite_median,
            ),
        )
        .reset_index()
    )

    display(
        band_summary
    )

    display(
        Markdown(
            f"**LSST DiaSource rows (up to {max_alert_rows})**"
        )
    )

    alert_cols = [
        "locus_id",
        "alert_id",
        "mjd",
        "band",
        "science_flux",
        "psf_flux",
        "ap_flux",
        "ap_psf_abs_fracdiff",
        "extendedness_for_summary",
        "src_e1",
        "src_e2",
        "src_e",
        "psf_e1",
        "psf_e2",
        "psf_e",
        "delta_e_for_summary",
        "abs_delta_e_for_summary",
        "diff_psf_snr",
    ]

    display(
        detail[
            alert_cols
        ].head(
            max_alert_rows
        )
    )


known_matches = cand[
    cand["known_lens_match"]
].copy()

if MAX_MATCH_DETAILS is not None:
    known_matches_for_details = (
        known_matches
        .head(
            MAX_MATCH_DETAILS
        )
    )
else:
    known_matches_for_details = (
        known_matches
    )

print(
    "Known-lens matches available for detail display: "
    f"{len(known_matches_for_details):,}"
)

if SHOW_MATCH_DETAILS:
    for _, row in known_matches_for_details.iterrows():
        show_matched_target_details(
            row
        )

## 12. Initialize the DP2 cutout service

In [ ]:
discovery = RSPDiscovery(
    "dp2"
)

sia_client = (
    discovery
    .get_sia_client()
)

## 13. ZIR cutout helpers with persistent caching

In [ ]:
# Do not erase already-downloaded cutouts if this cell is rerun.
if "CUTOUT_CACHE" not in globals():
    CUTOUT_CACHE = {}

if "SIA_CACHE" not in globals():
    SIA_CACHE = {}


def get_sia_results(
    ra,
    dec,
    radius=CUTOUT_RADIUS,
):
    key = (
        round(
            float(ra),
            8,
        ),
        round(
            float(dec),
            8,
        ),
        round(
            float(radius),
            8,
        ),
    )

    if key not in SIA_CACHE:
        SIA_CACHE[key] = sia_client.search(
            pos=(
                ra,
                dec,
                radius,
            ),
            calib_level=3,
            dpsubtype="lsst.deep_coadd",
        )

    return SIA_CACHE[key]


def get_dl_result(
    band,
    ra,
    dec,
    radius=CUTOUT_RADIUS,
):
    results = get_sia_results(
        ra,
        dec,
        radius=radius,
    )

    band_arr = np.asarray(
        results["lsst_band"]
    ).astype(str)

    indices = np.where(
        band_arr == band
    )[0]

    if len(indices) == 0:
        print(
            f"Band {band} does not exist at "
            f"RA={ra}, Dec={dec}."
        )
        return None

    index = int(
        indices[-1]
    )

    return (
        discovery
        .get_datalink_results(
            results[index]
        )
    )


def get_cutout(
    band,
    ra,
    dec,
    radius=CUTOUT_RADIUS,
):
    key = (
        band,
        round(
            float(ra),
            8,
        ),
        round(
            float(dec),
            8,
        ),
        round(
            float(radius),
            8,
        ),
    )

    if key in CUTOUT_CACHE:
        return CUTOUT_CACHE[key]

    dl_result = get_dl_result(
        band,
        ra,
        dec,
        radius=radius,
    )

    if dl_result is None:
        CUTOUT_CACHE[key] = None
        return None

    sq = SodaQuery.from_resource(
        dl_result,
        dl_result.get_adhocservice_by_id(
            "cutout-sync"
        ),
        session=get_pyvo_auth(),
    )

    sq.circle = (
        ra * u.deg,
        dec * u.deg,
        radius * u.deg,
    )

    cutout_bytes = (
        sq
        .execute_stream()
        .read()
    )

    sq.raise_if_error()

    cutout = read_archive(
        io.BytesIO(
            cutout_bytes
        )
    )

    CUTOUT_CACHE[key] = cutout

    return cutout


def normalize_band(
    image,
    asinh_a=ASINH_A,
    vmin=VMIN,
    vmax=VMAX,
):
    data = image.array

    norm = ImageNormalize(
        data,
        vmin=vmin,
        vmax=vmax,
        stretch=AsinhStretch(
            a=asinh_a
        ),
        clip=True,
    )

    return norm(data)


def combine_rgb(
    red_image,
    green_image,
    blue_image,
):
    red = normalize_band(
        red_image
    )

    green = normalize_band(
        green_image
    )

    blue = normalize_band(
        blue_image
    )

    return np.dstack(
        [
            red,
            green,
            blue,
        ]
    )


def fetch_zir_cutouts(
    ra,
    dec,
    radius=CUTOUT_RADIUS,
):
    cutouts = {}

    for band in (
        "r",
        "i",
        "z",
    ):
        try:
            cutouts[band] = get_cutout(
                band,
                ra,
                dec,
                radius=radius,
            )
        except Exception as e:
            print(
                f"Failed to fetch {band}-band cutout "
                f"for RA={ra}, Dec={dec}: {e}"
            )
            cutouts[band] = None

    return cutouts

## 14. Candidate text and ZIR display

In [ ]:
def fmt_value(
    value,
    fmt=".3f",
):
    try:
        if value is None or pd.isna(value):
            return "nan"
    except Exception:
        pass

    try:
        return format(
            value,
            fmt,
        )
    except Exception:
        return str(value)


def yesno(
    value,
):
    try:
        if pd.isna(value):
            return "nan"
    except Exception:
        pass

    return (
        "Y"
        if bool(value)
        else "N"
    )


def candidate_header(
    row,
):
    return (
        f"rank={int(row['rank'])}"
        f" | target_id={row['target_id']}"
        f" | score={fmt_value(row['second_pass_score'], '.3f')}"
    )


def candidate_subtext(
    row,
):
    lines = []

    lines.append(
        f"RA={fmt_value(row.get('ra'), '.6f')}  "
        f"Dec={fmt_value(row.get('dec'), '.6f')}"
    )

    lines.append(
        locus_summary_text(
            row
        )
    )

    mags = []

    for col in [
        "u_mag",
        "g_mag",
        "r_mag",
        "i_mag",
        "z_mag",
        "y_mag",
    ]:
        if col in row.index:
            mags.append(
                f"{col}="
                f"{fmt_value(row.get(col), '.2f')}"
            )

    if mags:
        lines.append(
            "  ".join(
                mags
            )
        )

    if "gaia_bp_rp_max" in row.index:
        lines.append(
            "BP-RP="
            f"{fmt_value(row.get('gaia_bp_rp_max'), '.2f')}"
        )

    lines.append(
        "ext: "
        f"med={fmt_value(row.get('extendedness_median'), '.3f')}  "
        f"max={fmt_value(row.get('extendedness_max'), '.3f')}  "
        f"f(>0.5)={fmt_value(row.get('extendedness_frac_gt_0p5'), '.2f')}  "
        f"N={fmt_value(row.get('extendedness_n'), '.0f')}"
    )

    lines.append(
        "ap-PSF: "
        f"median |dF|/|PSF|="
        f"{fmt_value(row.get('ap_psf_abs_fracdiff_median'), '.3f')}  "
        f"N={fmt_value(row.get('ap_psf_abs_fracdiff_n'), '.0f')}"
    )

    lines.append(
        "ellipticity: "
        f"median Δe={fmt_value(row.get('delta_e_median'), '.3f')}  "
        f"median |Δe|={fmt_value(row.get('abs_delta_e_median'), '.3f')}  "
        f"N={fmt_value(row.get('delta_e_n'), '.0f')}"
    )

    flags = []

    if "known_lens_match" in row.index:
        flags.append(
            "KnownLens="
            f"{yesno(row['known_lens_match'])}"
        )

    if (
        "known_lens_name" in row.index
        and str(
            row["known_lens_name"]
        ).strip()
    ):
        flags.append(
            "LensName="
            f"{row['known_lens_name']}"
        )

    if (
        "known_lens_sep_arcsec" in row.index
        and pd.notna(
            row["known_lens_sep_arcsec"]
        )
    ):
        flags.append(
            "LensSep="
            f"{fmt_value(row['known_lens_sep_arcsec'], '.2f')} arcsec"
        )

    if "milliquas_match" in row.index:
        flags.append(
            "MILLIQUAS="
            f"{yesno(row['milliquas_match'])}"
        )

    if "gaia_in_qso_candidates_any" in row.index:
        flags.append(
            "Gaia_QSO="
            f"{yesno(row['gaia_in_qso_candidates_any'])}"
        )

    if "passes_vetting" in row.index:
        flags.append(
            "PassesVetting="
            f"{yesno(row['passes_vetting'])}"
        )

    if flags:
        lines.append(
            "  ".join(
                flags
            )
        )

    return "\n".join(
        lines
    )


def show_candidate(
    row,
    radius=CUTOUT_RADIUS,
    figsize=FIGSIZE,
    skip_missing=SKIP_MISSING,
    show=True,
):
    ra = float(
        row["ra"]
    )

    dec = float(
        row["dec"]
    )

    cut = fetch_zir_cutouts(
        ra,
        dec,
        radius=radius,
    )

    need_zir = [
        cut.get("z"),
        cut.get("i"),
        cut.get("r"),
    ]

    if any(
        image is None
        for image in need_zir
    ):
        msg = (
            "Missing one or more ZIR bands for "
            f"{row['target_id']}"
        )

        if skip_missing:
            print(msg)
        else:
            raise RuntimeError(
                msg
            )

    fig, ax = plt.subplots(
        figsize=figsize
    )

    if all(
        image is not None
        for image in need_zir
    ):
        rgb_zir = combine_rgb(
            cut["z"],
            cut["i"],
            cut["r"],
        )

        ax.imshow(
            rgb_zir,
            origin="lower",
        )

    else:
        ax.text(
            0.5,
            0.5,
            "ZIR unavailable",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )

    ax.set_title(
        "ZIR"
    )

    ax.set_xticks([])
    ax.set_yticks([])

    edge_color = (
        "red"
        if bool(
            row.get(
                "known_lens_match",
                False,
            )
        )
        else "black"
    )

    for spine in ax.spines.values():
        spine.set_edgecolor(
            edge_color
        )
        spine.set_linewidth(
            2.0
            if edge_color == "red"
            else 1.0
        )

    fig.suptitle(
        candidate_header(
            row
        ),
        y=0.98,
        fontsize=11,
    )

    fig.text(
        0.02,
        0.02,
        candidate_subtext(
            row
        ),
        fontsize=8.2,
        va="bottom",
        ha="left",
    )

    legacy_url = row[
        "legacy_viewer_url"
    ]

    link_artist = fig.text(
        0.02,
        0.285,
        "Legacy Survey viewer",
        fontsize=9,
        color="blue",
        va="bottom",
        url=legacy_url,
    )

    link_artist.set_url(
        legacy_url
    )

    fig.subplots_adjust(
        bottom=0.46,
        top=0.90,
        left=0.06,
        right=0.98,
    )

    if show:
        display(
            Markdown(
                f"[Open Legacy Survey viewer]({legacy_url})"
            )
        )

        plt.show()

    return (
        fig,
        cut,
    )

## 15. Show all known-lens ZIR cutouts

In [ ]:
matched_for_cutouts = cand[
    cand["known_lens_match"]
].copy()

if MAX_MATCHED_TO_SHOW is not None:
    matched_for_cutouts = (
        matched_for_cutouts
        .head(
            MAX_MATCHED_TO_SHOW
        )
    )

print(
    "Known-lens cutouts to show: "
    f"{len(matched_for_cutouts):,}"
)

if SHOW_MATCHED_CUTOUTS:
    for _, row in matched_for_cutouts.iterrows():
        show_candidate(
            row
        )

## 16. Optional preview of the currently selected candidate sample

In [ ]:
subset = selected.iloc[
    START:START + N_SHOW
].copy()

preview_cols = [
    c
    for c in [
        "rank",
        "target_id",
        "second_pass_score",
        "ra",
        "dec",
        "gaia_bp_rp_max",
        "i_mag",
        "extendedness_median",
        "ap_psf_abs_fracdiff_median",
        "delta_e_median",
        "known_lens_match",
        "known_lens_name",
        "known_lens_sep_arcsec",
        "n_member_loci",
        "member_locus_ids",
        "first_cut_max_score",
        "n_tagged",
        "n_lsst_alerts",
        "legacy_viewer_url",
    ]
    if c in subset.columns
]

display(
    subset[
        preview_cols
    ]
)

if SHOW_PREVIEW:
    for _, row in subset.iterrows():
        show_candidate(
            row
        )
else:
    print(
        "SHOW_PREVIEW=False; no general DP2 cutouts requested."
    )

## 17. Optional multi-page PDF — disabled by default

In [ ]:
pdf_path = (
    OUTPUT_DIR
    / "candidate_rgb_vetting.pdf"
)

if SAVE_PDF:
    if (
        pdf_path.exists()
        and not OVERWRITE_OUTPUT
    ):
        raise FileExistsError(
            f"{pdf_path} exists and OVERWRITE_OUTPUT=False."
        )

    print(
        "Warning: bulk DP2 cutout requests may trigger HTTP 429 rate limits. "
        "SAVE_PDF is False by default."
    )

    with PdfPages(
        pdf_path
    ) as pdf:
        for _, row in selected.iterrows():
            try:
                fig, _ = show_candidate(
                    row,
                    show=False,
                )
            except Exception as e:
                fig, ax = plt.subplots(
                    figsize=(6, 3)
                )

                ax.axis(
                    "off"
                )

                ax.text(
                    0.5,
                    0.5,
                    (
                        f"{row['target_id']}\n\n"
                        "Failed to generate figure:\n"
                        f"{e}"
                    ),
                    ha="center",
                    va="center",
                    fontsize=10,
                )

            pdf.savefig(
                fig,
                bbox_inches="tight",
            )

            plt.close(
                fig
            )

    print(
        "Saved:",
        pdf_path,
    )
else:
    print(
        "SAVE_PDF=False; skipping bulk PDF generation."
    )

## Notes

- The current default cuts are:
  - `extendedness_median <= 0.50`;
  - `median(|apFlux-psfFlux|/|psfFlux|) <= 0.30`.
- Missing morphology measurements are retained rather than automatically rejected.
- `delta_e` is defined exactly as discussed:
  `e_source - e_PSF`, where `e = sqrt(e1**2 + e2**2)`.
- The correct alert-property moment keys are
  `ixx`, `iyy`, `ixy`, `ixxPSF`, `iyyPSF`, `ixyPSF`.
- Known-lens matches are displayed from the **pre-cut** candidate catalog and are annotated with
  `PassesVetting=Y/N`. This lets known lenses test the impact of the cuts.
- ZIR cutouts are cached in memory. Re-running the helper cell does **not** clear the cache.
- The SIA search itself is also cached once per sky position.
- PDF generation is off by default to avoid repeated requests and HTTP 429 rate limiting.